# Rebuild and evaluate the publication QUBOs

`qubo_tools.py` is standalone and uses only the Python standard library. The three public workflow functions are `build_sub_QUBO`, `build_QUBO`, and `evaluate_solution`. Start Jupyter from this repository's root. If the kernel starts in `code/`, setup moves its working directory to the repository root; all data paths below are relative to that root.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Jupyter may start the kernel in the notebook's code/ directory.
if Path.cwd().name == "code" and Path("qubo_tools.py").is_file():
    os.chdir(Path.cwd().parent)
REPO_ROOT = Path(".")
if not all((REPO_ROOT / name).is_dir() for name in ("code", "instances", "results")):
    raise FileNotFoundError("Start Jupyter from this repository's root directory.")
CODE_ROOT = REPO_ROOT / "code"
if str(CODE_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT.resolve()))
from qubo_tools import PART_DESCRIPTIONS, build_sub_QUBO, build_QUBO, evaluate_solution

# All data paths are relative to this repository's root.

instance_path = REPO_ROOT / "instances" / "p_ucp_u2_t3_v1"
result_path = (
    REPO_ROOT
    / "results"
    / "fix_multiobj"
    / "p_ucp_fix_multiobj_u2_t3_v1.json"
)
solution_name = "best_original_solution_overall"

# 1. Build the six unweighted sub-QUBOs.

In [ ]:
sub_qubos = build_sub_QUBO(instance_path)
for label, sub_qubo in sub_qubos.items():
    print(label, "-", PART_DESCRIPTIONS[label], sub_qubo.summary())

# 2. Build the full QUBO with the factors stored for one solution.

In [ ]:
qubo = build_QUBO(instance_path, result_path, solution_name)
qubo.summary()

# 3. Read that solution and evaluate it on the reconstructed QUBO.

In [ ]:
with result_path.open(encoding="utf-8") as file:
    result = json.load(file)

calculated = evaluate_solution(qubo, result, solution_name)
reported = result[solution_name]["QUBO_value"]
print(f"calculated = {calculated}")
print(f"reported   = {reported}")
print(f"difference = {calculated - reported}")

# Optional: reproduce the unweighted split energies.

In [ ]:

solution = result[solution_name]
state = solution["decision_variables"]
calculated_parts = {
    label: part.evaluate(state, strict=False)
    for label, part in sub_qubos.items()
}
if "c+d" in solution["split_energies"]:
    calculated_parts["c+d"] = calculated_parts["c"] + calculated_parts["d"]
calculated_parts